# 02c — Recompute Social Optimum only (1200 s limit)

Re-solves **only** the social-optimum reference for every layered NFG instance
with a 1,200 s time limit, then patches the `so_*` and `pos` columns of the
existing Experiment-1 result CSVs **in place**.

BRD and GZR results are left untouched: the social optimum is an independent
MIP over the same feasible region and does not depend on them (no warm start
is used), so re-running it alone is sound and keeps each CSV internally
consistent.

Patches both `nfg_layered_exp1_brd_gzr_pos.csv` (VEST on) and
`nfg_layered_exp1b_no_vest.csv` (VEST off) if present. Originals are backed up
with a `.bak_so600` suffix before being overwritten.

Afterwards run `python ../make_nfg_tables.py` to regenerate every NFG table.

In [ ]:
import json, glob, time, shutil
import numpy as np
import pandas as pd
from pathlib import Path

from gipg.nfg.instance import NFGInstance
from gipg.nfg.social_optimum import solve_social_optimum, compute_pos

RESULTS_DIR = Path('../results')
SHARED_DIR  = Path('../data/nfg')
SO_TIME_LIMIT = 1200.0   # 20 minutes (was 600 s)

assert SHARED_DIR.exists(), f'instances not found at {SHARED_DIR}'
print('modules loaded | SO_TIME_LIMIT =', SO_TIME_LIMIT)

In [ ]:
# --- load layered instances (identical to 02_run_nfg_layered) ---
json_files = sorted(glob.glob(str(SHARED_DIR / 'E_*.json'))) + \
             sorted(glob.glob(str(SHARED_DIR / 'F_*.json')))

instances = {}
for fp in json_files:
    with open(fp) as f:
        d = json.load(f)
    instances[d['tag']] = NFGInstance(
        n_nodes=d['n_nodes'],
        edges=tuple(tuple(e) for e in d['edges']),
        n_players=d['n_players'],
        sources=np.array(d['sources'], dtype=int),
        sinks=np.array(d['sinks'], dtype=int),
        demands=np.array(d['demands'], dtype=int),
        capacities=np.array(d['capacities'], dtype=int),
        edge_costs=np.array(d['edge_costs'], dtype=float),
        seed=d['seed'],
        meta=d['meta'],
    )
print(f'Loaded {len(instances)} layered instances')

In [ ]:
# --- solve SO once per instance ---
so_rows = []
t_start = time.time()

for k, (tag, inst) in enumerate(sorted(instances.items()), 1):
    t0 = time.time()
    res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
    so_rows.append({
        'tag': tag,
        'so_status': res.status,
        'so_cost': res.opt_cost,
        'so_time': res.runtime,
    })
    flag = '' if res.status == 'OPTIMAL' else f'  <-- {res.status}'
    print(f'[{k:3d}/{len(instances)}] {tag:38s} {res.status:9s} '
          f'cost={res.opt_cost} {time.time()-t0:7.1f}s{flag}', flush=True)

so_df = pd.DataFrame(so_rows)
print(f'\nTotal SO wall-clock: {(time.time()-t_start)/60:.1f} min')
print(so_df['so_status'].value_counts().to_string())
so_df.to_csv(RESULTS_DIR / 'nfg_layered_so_1200s.csv', index=False)
print(f"\nSaved -> {RESULTS_DIR / 'nfg_layered_so_1200s.csv'}")

In [ ]:
# --- patch the Experiment-1 CSVs in place ---
targets = ['nfg_layered_exp1_brd_gzr_pos.csv', 'nfg_layered_exp1b_no_vest.csv']
so_map = so_df.set_index('tag')

for name in targets:
    path = RESULTS_DIR / name
    if not path.exists():
        print(f'skip (not found): {name}')
        continue

    df = pd.read_csv(path)
    if 'best_pne_cost' not in df.columns:
        print(f'skip ({name}): no best_pne_cost column -- nothing to recompute')
        continue

    shutil.copy(path, path.with_suffix('.csv.bak_so600'))

    key = df['tag'].str.replace('.json', '', regex=False)
    old_status = df['so_status'].copy() if 'so_status' in df.columns else None

    df['so_status'] = key.map(so_map['so_status'])
    df['so_cost']   = key.map(so_map['so_cost'])
    df['so_time']   = key.map(so_map['so_time'])
    df['pos'] = [
        compute_pos(e, s) if pd.notna(e) and pd.notna(s) else np.nan
        for e, s in zip(df['best_pne_cost'], df['so_cost'])
    ]

    df.to_csv(path, index=False)

    n_bad = int((pd.to_numeric(df['pos'], errors='coerce') < 1 - 1e-9).sum())
    n_nc  = int((df['so_status'] != 'OPTIMAL').sum())
    was   = int((old_status != 'OPTIMAL').sum()) if old_status is not None else -1
    print(f'{name}')
    print(f'   backup      -> {path.name}.bak_so600')
    print(f'   SO non-opt  : {was} (600 s)  ->  {n_nc} (1200 s)')
    print(f'   POS < 1     : {n_bad}')

In [ ]:
# --- final check ---
print('Next: python ../make_nfg_tables.py\n')
for name in targets:
    p = RESULTS_DIR / name
    if not p.exists():
        continue
    d = pd.read_csv(p)
    pos = pd.to_numeric(d['pos'], errors='coerce')
    ok  = d['so_status'].eq('OPTIMAL')
    print(f'{name:38s} rows={len(d)}  SO-opt={int(ok.sum())}  '
          f'POS<1={int((pos < 1 - 1e-9).sum())}  '
          f'POS(mean, SO-opt only)={pos[ok].mean():.4f}')